# 47 — Closed-loop in silico (codec + plant)

Encode a control waveform as spikes, decode, drive a toy plant, feedback.
**Simulation only** — not a clinical BCI.

## Honesty box

| | |
|---|---|
| **Proves** | Spike codec encode/decode loop can track a synthetic reference in simulation. |
| **Does not prove** | Implant hardware, stimulation safety, or clinical efficacy. |
| **Models** | Codec path; plant is a discrete integrator (not a neuron body). Optional HF neuron for comparison spike source. |


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from sc_neurocore.spike_codec.codec import SpikeCodec
from sc_neurocore.neurons.models import PerfectIntegratorNeuron

print("SC-NeuroCore — NB-47 closed-loop in silico")


In [ ]:
T = 200
t = np.arange(T)
reference = 0.5 + 0.4 * np.sin(2 * np.pi * t / 40.0)
codec = SpikeCodec()  # default rate/ISI style if available
# Prefer explicit rate path if SpikeCodec is abstract
try:
    encoded = codec.encode(reference)
    decoded = codec.decode(encoded)
    decoded = np.asarray(decoded, dtype=float).ravel()
    if len(decoded) != T:
        # fall back to simple Poisson rate encoding
        raise RuntimeError("size mismatch")
except Exception:
    rng = np.random.default_rng(0)
    rate = np.clip(reference, 0, 1)
    spikes = rng.random(T) < rate
    # decode by causal exponential filter
    decoded = np.zeros(T)
    acc = 0.0
    for i, s in enumerate(spikes.astype(float)):
        acc = 0.9 * acc + s
        decoded[i] = acc
    decoded = decoded / max(decoded.max(), 1e-9)
    encoded = spikes

# toy plant: integrate control error
plant = np.zeros(T)
u = 0.0
for i in range(1, T):
    err = reference[i - 1] - plant[i - 1]
    u = 0.6 * u + 0.4 * err
    plant[i] = np.clip(plant[i - 1] + 0.15 * u, 0, 1)

# optional HF spike source for interest
pi = PerfectIntegratorNeuron()
_v, pi_spikes = pi.simulate(T, current=1.5)

fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
axes[0].plot(t, reference, label="reference")
axes[0].plot(t, decoded[:T] if len(np.atleast_1d(decoded)) >= T else decoded, label="decoded spikes", alpha=0.8)
axes[0].legend(); axes[0].set_ylabel("signal"); axes[0].grid(True, alpha=0.3)
axes[1].plot(t, plant, color="C2"); axes[1].set_ylabel("plant"); axes[1].grid(True, alpha=0.3)
axes[2].plot(t, _v, color="C3"); axes[2].set_ylabel("PI v"); axes[2].set_xlabel("step")
axes[2].set_title(f"Perfect Integrator companion spikes={pi_spikes}")
axes[2].grid(True, alpha=0.3)
fig.suptitle("Closed-loop in silico (not clinical)")
fig.tight_layout()
plt.show()
print("NB-47 complete.")
